In [1]:
import os
import json
from openai import OpenAI
from google.colab import userdata

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from uuid import uuid4 as uuid

In [2]:
# !pip install langchain_openai
# !pip install langchain_chroma

## LangChain Introduction

In [3]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage  # Chat history message types

In [4]:
embedding_model = OpenAIEmbeddings(
    model=os.getenv('OPENAI_EMBEDDING_MODEL', 'text-embedding-3-small'), api_key=userdata.get('openai_IK')
)
print("✓ OpenAI embedding model ready")

✓ OpenAI embedding model ready


### single text  embedding vs multi texts embeddings

In [ ]:
embeddings = embedding_model.embed_query("Hello")
print(np.array(embeddings).shape)

(1536,)


In [ ]:
embeddings = embedding_model.embed_documents(["Hello", "world"])
print(np.array(embeddings).shape)

(2, 1536)


### embedding into vector db

In [ ]:
print(os.listdir('/content/sample_data'))

['.ipynb_checkpoints', 'synthetic_tickets.json']


In [5]:
with open('./sample_data/synthetic_tickets.json', 'r') as f:
    data = json.load(f)

In [ ]:
data[:1]

[{'ticket_id': 'TICK-001',
  'title': 'Users unable to log in after password reset',
  'description': "Multiple users reporting authentication failures after performing password reset. Error message: 'Invalid credentials'. Issue started after recent security patch deployment.",
  'resolution': "Found that the password hash algorithm was updated but session tokens weren't invalidated. Solution: Clear all active sessions and force re-authentication. Implemented automatic session cleanup on password change.",
  'category': 'Authentication',
  'priority': 'High',
  'created_date': '2024-01-15',
  'resolved_date': '2024-01-15'}]

In [6]:
documents = []

for ticket in data:
      content = f"""
        Ticket ID: {ticket['ticket_id']}
        Title: {ticket['title']}
        Category: {ticket['category']}
        Priority: {ticket['priority']}
        Date: {ticket['created_date']} to {ticket['resolved_date']}

        Problem Description:
        {ticket['description']}

        Resolution:
        {ticket['resolution']}
      """.strip()

      metadata = {
          'ticket_id': ticket['ticket_id'],
          'title': ticket['title'],
          'category': ticket['category'],
          'priority': ticket['priority'],
          'created_date': ticket['created_date'],
          'resolved_date': ticket['resolved_date'],
      }


      document = Document(page_content=content, metadata=metadata)
      documents.append(document)


vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name="supportdesk_rag",
    persist_directory="./rag_vectorstore",
    collection_metadata={"hnsw:space": "cosine"}
)

In [ ]:
# Access the underlying Chroma client collection
chroma_collection = vector_store._collection

# Retrieve the first few documents (e.g., 5 documents)
# 'documents' here refers to the page_content of the stored Document
retrieved_data = chroma_collection.get(
    limit=5,
    include=['metadatas', 'documents'] # Specify what to include in the retrieval
)

print("Retrieved documents from ChromaDB:")
for i in range(len(retrieved_data['ids'][:2])):
    print(f"--- Document {i+1} ---")
    print(f"ID: {retrieved_data['ids'][i]}")
    print(f"Metadata: {retrieved_data['metadatas'][i]}")
    print(f"Page Content (truncated): {retrieved_data['documents'][i][:200]}...")
    print("-" * 20)

Retrieved documents from ChromaDB:
--- Document 1 ---
ID: 7bf564d7-b25f-426e-8990-2bae5ad307c7
Metadata: {'title': 'Users unable to log in after password reset', 'created_date': '2024-01-15', 'ticket_id': 'TICK-001', 'resolved_date': '2024-01-15', 'priority': 'High', 'category': 'Authentication'}
Page Content (truncated): Ticket ID: TICK-001
        Title: Users unable to log in after password reset
        Category: Authentication
        Priority: High
        Date: 2024-01-15 to 2024-01-15

        Problem Descripti...
--------------------
--- Document 2 ---
ID: 3bc27b7d-f804-4de0-928d-b01c7cb734b8
Metadata: {'ticket_id': 'TICK-002', 'title': 'Database connection timeout in production', 'resolved_date': '2024-01-18', 'priority': 'Critical', 'category': 'Database', 'created_date': '2024-01-18'}
Page Content (truncated): Ticket ID: TICK-002
        Title: Database connection timeout in production
        Category: Database
        Priority: Critical
        Date: 2024-01-18 to 2024-01

In [13]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

user_query = "Database is timing out frequently"

response = retriever.invoke(user_query)

for doc in response:
    print(doc)
    print(doc.metadata)
    print("="*50)

page_content='Ticket ID: TICK-002
        Title: Database connection timeout in production
        Category: Database
        Priority: Critical
        Date: 2024-01-18 to 2024-01-18

        Problem Description:
        Application experiencing intermittent 500 errors. Logs show 'connection pool exhausted' and 'timeout waiting for connection'. Affects approximately 20% of requests during peak hours.

        Resolution:
        Database connection pool was sized too small for peak load. Increased max_connections from 20 to 100. Added connection pooling monitoring and alerts. Optimized long-running queries that were holding connections.' metadata={'ticket_id': 'TICK-002', 'resolved_date': '2024-01-18', 'created_date': '2024-01-18', 'category': 'Database', 'priority': 'Critical', 'title': 'Database connection timeout in production'}
{'ticket_id': 'TICK-002', 'resolved_date': '2024-01-18', 'created_date': '2024-01-18', 'category': 'Database', 'priority': 'Critical', 'title': 'Database c

In [7]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
print(format_docs(response))

Ticket ID: TICK-002
        Title: Database connection timeout in production
        Category: Database
        Priority: Critical
        Date: 2024-01-18 to 2024-01-18

        Problem Description:
        Application experiencing intermittent 500 errors. Logs show 'connection pool exhausted' and 'timeout waiting for connection'. Affects approximately 20% of requests during peak hours.

        Resolution:
        Database connection pool was sized too small for peak load. Increased max_connections from 20 to 100. Added connection pooling monitoring and alerts. Optimized long-running queries that were holding connections.

Ticket ID: TICK-014
        Title: Session timeout too short
        Category: Authentication
        Priority: Medium
        Date: 2024-02-16 to 2024-02-17

        Problem Description:
        Users complaining about being logged out too frequently. Currently set to 15 minutes. Particularly problematic for users filling out long forms, losing unsaved data when s

In [8]:
prompt_template = """You are SupportDesk AI, a technical support assistant that helps engineers troubleshoot issues using historical support ticket data.

CRITICAL RULES:
1. Answer using ONLY information from the provided context.
2. If the question is broad or underspecified, provide the best matching known issue(s) from context and state any assumptions.
3. If context is partially relevant, still provide the most likely troubleshooting guidance from relevant tickets.
4. If the answer is truly not present in context, say "I don't have enough information in the ticket history to answer that question."
5. DO NOT make up information or use external knowledge.
6. Always cite ticket IDs for every issue/resolution you mention.
7. If multiple tickets are relevant, summarize each briefly.

Context from support tickets:
{context}

Question: {question}

Helpful Answer (with ticket citations):"""

PROMPT = ChatPromptTemplate.from_template(prompt_template)

In [ ]:
PROMPT

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='You are SupportDesk AI, a technical support assistant that helps engineers troubleshoot issues using historical support ticket data.\n\nCRITICAL RULES:\n1. Answer using ONLY information from the provided context.\n2. If the question is broad or underspecified, provide the best matching known issue(s) from context and state any assumptions.\n3. If context is partially relevant, still provide the most likely troubleshooting guidance from relevant tickets.\n4. If the answer is truly not present in context, say "I don\'t have enough information in the ticket history to answer that question."\n5. DO NOT make up information or use external knowledge.\n6. Always cite ticket IDs for every issue/resolution you mention.\n7. If multiple tickets are releva

In [9]:
llm = ChatOpenAI(
    model='gpt-4o-mini',
    temperature=0,  # Temperature controls randomness (0 = deterministic, 2 = very creative)
    timeout=120,  # Increase timeout for slower connections
    max_retries=3,  # Retry on transient failures
    api_key=userdata.get('openai_IK')
)

In [14]:
chain = {
  "context": retriever | format_docs,
  "question": RunnablePassthrough()
} | PROMPT | llm | StrOutputParser()

How chaining will internally work



```
question = user_input

docs = retriever.invoke(question)

context = format_docs(docs)

prompt_input = {
    "context": context,
    "question": question
}

prompt_text = PROMPT.invoke(prompt_input)

response = llm.invoke(prompt_text)

answer = StrOutputParser().invoke(response)
```



In [15]:
test_queries = [
    "How do I fix authentication failures after password reset?",
    "What causes database connection timeouts?",
    "Why are emails not being delivered?",
    "How do I make the perfect pizza?"  # Should refuse to answer!
]

for query in test_queries:
  result = chain.invoke(query)

  print("\n" + "-"*80)
  print("ANSWER:")
  print("-"*80)
  print(result)


--------------------------------------------------------------------------------
ANSWER:
--------------------------------------------------------------------------------
To fix authentication failures after a password reset, you can refer to the resolution provided in Ticket ID: TICK-001. The issue was caused by an updated password hash algorithm while the session tokens remained active, leading to 'Invalid credentials' errors for users.

**Resolution Steps:**
1. Clear all active sessions for users who have recently reset their passwords.
2. Force re-authentication for these users.
3. Implement automatic session cleanup on password changes to prevent similar issues in the future.

By following these steps, you should be able to resolve the authentication failures after a password reset.

--------------------------------------------------------------------------------
ANSWER:
--------------------------------------------------------------------------------
Database connection timeouts c

## Implementing Chunking+RAG with langchain

In [ ]:
# pip install -U langchain-text-splitters

In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [18]:
documents = []

for ticket in data:
      content = f"""
        Ticket ID: {ticket['ticket_id']}
        Title: {ticket['title']}
        Category: {ticket['category']}
        Priority: {ticket['priority']}
        Date: {ticket['created_date']} to {ticket['resolved_date']}

        Problem Description:
        {ticket['description']}

        Resolution:
        {ticket['resolution']}
      """.strip()

      metadata = {
          'ticket_id': ticket['ticket_id'],
          'title': ticket['title'],
          'category': ticket['category'],
          'priority': ticket['priority'],
          'created_date': ticket['created_date'],
          'resolved_date': ticket['resolved_date'],
      }


      document = Document(page_content=content, metadata=metadata)
      documents.append(document)

In [19]:
splitter = RecursiveCharacterTextSplitter(
    separators = ["\n\n", "\n", " "],
    chunk_size = 450,
    chunk_overlap  = 20,
    length_function = len
)

chunks = splitter.split_documents(documents)

In [20]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="supportdesk_rag_chunk",
    persist_directory="./rag_vectorstore",
    collection_metadata={"hnsw:space": "cosine"}
)

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

chain = {
  "context": retriever | format_docs,
  "question": RunnablePassthrough()
} | PROMPT | llm | StrOutputParser()

In [ ]:
test_queries = [
    "How do I fix authentication failures after password reset?",
    "What causes database connection timeouts?",
    "Why are emails not being delivered?",
    "How do I make the perfect pizza?"  # Should refuse to answer!
]

for query in test_queries:
  docs = retriever.invoke(query)
  result = chain.invoke(query)

  print("\n" + "-"*80)
  print("ANSWER:")
  print("-"*80)
  print(result)
  print("\n")
  print("Retrieved Context Metadata: ")

  for i, doc in enumerate(docs, 1):
    print(f"{i}. {doc.metadata['title']}")




--------------------------------------------------------------------------------
ANSWER:
--------------------------------------------------------------------------------
To address authentication failures after a password reset, you can refer to the issue documented in Ticket ID: TICK-001. The problem involved multiple users experiencing authentication failures with the error message 'Invalid credentials' following a recent security patch deployment.

Here are some steps you can take based on this ticket:

1. **Verify Password Reset Process**: Ensure that the password reset process is functioning correctly and that users are following the correct steps to reset their passwords.

2. **Check for Security Patch Issues**: Since the issue began after a recent security patch deployment, investigate if the patch introduced any bugs or changes that could affect the authentication process.

3. **User Account Status**: Confirm that the affected user accounts are active and not locked or disable

In [45]:
query = "How do I fix authentication failures after password reset?"

docs = retriever.invoke(query)
result = chain.invoke(query)

history = []

history.append(HumanMessage(content=query))
history.append(AIMessage(content=result))

In [25]:
history

[HumanMessage(content='How do I fix authentication failures after password reset?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="To fix authentication failures after a password reset, you can refer to the resolution from Ticket ID: TICK-001. The issue was caused by an updated password hash algorithm while the session tokens remained active, leading to 'Invalid credentials' errors for users.\n\nThe solution implemented was to clear all active sessions and force re-authentication. Additionally, it was recommended to implement automatic session cleanup upon password changes to prevent similar issues in the future. \n\nMake sure to follow these steps to resolve the authentication failures.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [46]:
condense_prompt = ChatPromptTemplate.from_messages([
    ("system",
      "Given the chat history and a follow-up question, rephrase the "
      "follow-up as a standalone question that includes all necessary "
      "context from the history. If the question is already standalone, "
      "return it unchanged."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}"),
])

condense_chain = {
    "chat_history": lambda x: history,
    "question": RunnablePassthrough()
} | condense_prompt | llm | StrOutputParser()

query = "What was the ticket ID for that issue?"

result = condense_chain.invoke(query)

history.append(HumanMessage(content=query))
history.append(AIMessage(content=result))

In [41]:
history

[HumanMessage(content='How do I fix authentication failures after password reset?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="To fix authentication failures after a password reset, you can refer to the following issue documented in ticket TICK-001. \n\nIn that case, multiple users reported authentication failures with the error message 'Invalid credentials' after performing a password reset. This issue arose after a recent security patch deployment, which updated the password hash algorithm but did not invalidate existing session tokens.\n\nThe resolution involved two key steps:\n1. Clear all active sessions to ensure that users are required to re-authenticate.\n2. Implement automatic session cleanup on password changes to prevent similar issues in the future.\n\nBy following these steps, you should be able to resolve authentication failures after a password reset.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(

In [42]:
print(result)

The ticket ID for that issue was TICK-001.


In [48]:
# context = format_docs(retriever.invoke(result))

In [49]:
conv_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are SupportDesk AI. Answer using the ticket context below and the chat history.
    Use chat history to resolve references like "that issue" or "that ticket".
    For factual claims, prioritize the retrieved context.
    If information is not available in context or history, say "I don't have that information."
    Always cite ticket IDs when available.

    Context:
    {context}"""),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{question}"),
])

conv_chain = {
    "context": lambda x: format_docs(retriever.invoke(result)),
    "chat_history": lambda x: history,
    "question": RunnablePassthrough(),
 } | conv_prompt| llm | StrOutputParser()

query = "What was the resolution for that ticket?"

result = conv_chain.invoke(query)

history.append(HumanMessage(content=query))
history.append(AIMessage(content=result))

In [50]:
print(result)

The resolution for Ticket ID TICK-001, which dealt with authentication failures after a password reset, involved the following steps:

1. Clearing all active user sessions.
2. Forcing users to re-authenticate.
3. Implementing an automatic session cleanup to occur whenever a password is changed.

These actions were taken to address the 'Invalid credentials' errors that users were experiencing.


## Interactive Demo

In [57]:
def ask_question(ques, history=[]):
  if len(history) == 0:
    chain = {
      "context": retriever | format_docs,
      "question": RunnablePassthrough()
    } | PROMPT | llm | StrOutputParser()

    result = chain.invoke(ques)
  else:
    condense_chain = {
      "chat_history": lambda x: history,
      "question": RunnablePassthrough()
    } | condense_prompt | llm | StrOutputParser()

    condense_result = condense_chain.invoke(ques) # re-phrase question

    conv_chain = {
        "context": lambda x: format_docs(retriever.invoke(condense_result)),
        "chat_history": lambda x: history,
        "question": RunnablePassthrough(),
    } | conv_prompt| llm | StrOutputParser()

    result = conv_chain.invoke(ques)

  history.append(HumanMessage(content=ques))
  history.append(AIMessage(content=result))

  return result, history

In [58]:
history = []

while True:

  if len(history) == 0:
    print("\nAssistant: How can I help !\n")

  user = input("You: ").strip()

  if user.lower() in ['q', 'exit']:
    break

  result, history = ask_question(user, history)
  print(f"\nAssistant: {result}\n")


Assistant: How can I help !

You: How do I fix authentication failures after a password reset?

Assistant: To fix authentication failures after a password reset, you can refer to the resolution from Ticket ID: TICK-001. The issue was caused by an updated password hash algorithm while the session tokens remained active, leading to 'Invalid credentials' errors for users.

The solution implemented was to clear all active sessions and force re-authentication. Additionally, it was recommended to implement automatic session cleanup upon password changes to prevent similar issues in the future. 

Make sure to follow these steps to resolve the authentication failures.

You: What was the ticket ID for that issue?

Assistant: The ticket ID for that issue is TICK-001.

You: What was the resolution for that ticket?

Assistant: The resolution for Ticket ID: TICK-001 involved clearing all active sessions and forcing re-authentication for users. It was also recommended to implement automatic session